# Week 4 Task 2 — Reusable Fare Feature Pipeline

## Goal

Transform the Task 1 fare training Delta table into a model-ready feature dataset with a reusable Spark ML pipeline.

This notebook:

- loads the chronological Train, Validation, and Test splits created in Task 1;
- handles missing numerical and categorical values;
- indexes and one-hot encodes categorical variables;
- robustly scales continuous numerical variables for multiple linear regression;
- assembles the final Spark ML `features` vector;
- fits every learned preprocessing stage on Train only;
- applies the fitted pipeline unchanged to Validation and Test;
- saves both the fitted feature pipeline and the final feature Delta table.

Model fitting is intentionally deferred to Task 3.

## Setup

The notebook locates the repository through `pyproject.toml`, so it remains portable after the project directory is moved.

In [1]:
from pathlib import Path
import sys

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer,
    OneHotEncoder,
    RobustScaler,
    SQLTransformer,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TASK1_PATH = PROJECT_ROOT / "data" / "processed" / "ml" / "fare_training_dataset"
FEATURE_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "ml" / "fare_feature_dataset"
PIPELINE_MODEL_PATH = PROJECT_ROOT / "data" / "models" / "fare_feature_pipeline"

source_package = str(PROJECT_ROOT / "src")
if source_package not in sys.path:
    sys.path.insert(0, source_package)

from data_intensive_computing.spark import build_spark_session

spark = build_spark_session(
    "Week4-Task2-Fare-Feature-Pipeline",
    driver_memory="8g",
    shuffle_partitions=32,
)

print("Project root:", PROJECT_ROOT)
print("Task 1 input:", TASK1_PATH)
print("Feature output:", FEATURE_OUTPUT_PATH)
print("Fitted pipeline:", PIPELINE_MODEL_PATH)

:: loading settings :: url = jar:file:/Users/chloexia/Documents/KTH/Period%201/Data-Intensive%20Computing/Labs/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/chloexia/.ivy2.5.2/cache
The jars for the packages stored in: /Users/chloexia/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3e121603-634b-4dc1-afac-f9428f47560e;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	fo

	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.0 in central
:: resolution report :: resolve 240ms :: artifacts dl 6ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 f

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/chloexia/Documents/KTH/Period 1/Data-Intensive Computing/Labs/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/09/29 19:40:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Project root: /Users/chloexia/Documents/KTH/Period 1/Data-Intensive Computing/Labs/data-intensive-computing
Task 1 input: /Users/chloexia/Documents/KTH/Period 1/Data-Intensive Computing/Labs/data-intensive-computing/data/processed/ml/fare_training_dataset
Feature output: /Users/chloexia/Documents/KTH/Period 1/Data-Intensive Computing/Labs/data-intensive-computing/data/processed/ml/fare_feature_dataset
Fitted pipeline: /Users/chloexia/Documents/KTH/Period 1/Data-Intensive Computing/Labs/data-intensive-computing/data/models/fare_feature_pipeline


### Key design decisions

- The pipeline is fitted only on `split = 'train'` to prevent leakage.
- `weather_snow_depth_mm` is excluded because it is entirely missing in Train.
- Sparse PM2.5 is retained with median imputation and the existing `air_quality_missing` indicator.
- Continuous variables are scaled separately from binary, cyclical, and one-hot features.
- `RobustScaler` is used instead of min-max normalization because distance contains extreme observations.
- One-hot encoding uses `dropLast=True`, which avoids a redundant dummy column for linear regression.
- The fare label is not scaled, so future MAE and RMSE remain interpretable in US dollars.
- Target outliers are reported but not silently removed or capped.

## Steps

### 1. Load and verify the Task 1 dataset

In [2]:
fare_data = spark.read.format("delta").load(str(TASK1_PATH))

required_columns = {
    "trip_id",
    "pickup_datetime",
    "split",
    "label",
    "pickup_zone",
    "dropoff_zone",
    "pickup_borough",
    "dropoff_borough",
    "trip_distance",
    "passenger_count",
    "vendor_id",
    "rate_code_id",
    "store_and_fwd_flag",
    "pickup_month_of_year",
    "is_weekend",
    "pickup_hour_sin",
    "pickup_hour_cos",
    "pickup_dow_sin",
    "pickup_dow_cos",
    "weather_temperature_c",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "weather_wind_speed_kmh",
    "weather_pressure_hpa",
    "weather_condition_code",
    "air_quality_pm25_mean",
    "air_quality_missing",
}

missing_columns = sorted(required_columns - set(fare_data.columns))
assert not missing_columns, f"Task 1 dataset is missing required columns: {missing_columns}"

input_split_counts = {
    row["split"]: row["count"]
    for row in fare_data.groupBy("split").count().collect()
}
assert set(input_split_counts) == {"train", "validation", "test"}
print("Input split counts:", input_split_counts)

26/09/29 19:40:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



[Stage 2:=================>                                      (10 + 10) / 32]



Input split counts: {'train': 9381837, 'test': 306990, 'validation': 167440}


### 2. Inspect Train-only missingness and target outliers

These checks document why snow depth is removed and why robust scaling is appropriate. No Validation or Test statistics are used to configure the pipeline.

In [3]:
train_raw = fare_data.filter(F.col("split") == "train")
validation_raw = fare_data.filter(F.col("split") == "validation")
test_raw = fare_data.filter(F.col("split") == "test")

missingness_columns = [
    "trip_distance",
    "passenger_count",
    "weather_temperature_c",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "weather_snow_depth_mm",
    "weather_wind_speed_kmh",
    "weather_pressure_hpa",
    "air_quality_pm25_mean",
]

missingness = train_raw.agg(
    *[
        F.round(
            100.0 * F.avg(F.col(column).isNull().cast("double")),
            2,
        ).alias(f"{column}_null_pct")
        for column in missingness_columns
    ]
)
missingness.show(truncate=False, vertical=True)

label_quantile_levels = [0.50, 0.95, 0.99, 0.999]
label_quantiles = train_raw.approxQuantile(
    "label",
    label_quantile_levels,
    relativeError=0.0001,
)
distance_quantiles = train_raw.approxQuantile(
    "trip_distance",
    label_quantile_levels,
    relativeError=0.0001,
)

print("Train label quantiles:", dict(zip(label_quantile_levels, label_quantiles)))
print("Train distance quantiles:", dict(zip(label_quantile_levels, distance_quantiles)))
print("Maximum Train fare:", train_raw.agg(F.max("label")).first()[0])
print("Maximum Train distance:", train_raw.agg(F.max("trip_distance")).first()[0])

-RECORD 0---------------------------------------
 trip_distance_null_pct                 | 0.0   
 passenger_count_null_pct               | 7.78  
 weather_temperature_c_null_pct         | 0.0   
 weather_relative_humidity_pct_null_pct | 0.0   
 weather_precipitation_mm_null_pct      | 7.47  
 weather_snow_depth_mm_null_pct         | 100.0 
 weather_wind_speed_kmh_null_pct        | 0.0   
 weather_pressure_hpa_null_pct          | 0.0   
 air_quality_pm25_mean_null_pct         | 90.01 




[Stage 18:===============>                                        (9 + 10) / 33]



Train label quantiles: {0.5: 13.5, 0.95: 60.0, 0.99: 74.4, 0.999: 131.8}
Train distance quantiles: {0.5: 1.7, 0.95: 12.99, 0.99: 19.84, 0.999: 29.03}


Maximum Train fare: 9792.0
Maximum Train distance: 312722.3


### 3. Define feature groups

Only continuous variables are robustly scaled. Binary values, sine/cosine encodings, and one-hot vectors keep their natural ranges.

In [4]:
imputer_input_cols = [
    "passenger_count",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "air_quality_pm25_mean",
]
imputer_output_cols = [f"{column}_imputed" for column in imputer_input_cols]

continuous_numeric_cols = [
    "trip_distance",
    "passenger_count_imputed",
    "pickup_month_of_year",
    "weather_temperature_c",
    "weather_relative_humidity_pct_imputed",
    "weather_precipitation_mm_imputed",
    "weather_wind_speed_kmh",
    "weather_pressure_hpa",
    "air_quality_pm25_mean_imputed",
]

binary_cols = [
    "is_weekend",
    "air_quality_missing",
]

cyclical_cols = [
    "pickup_hour_sin",
    "pickup_hour_cos",
    "pickup_dow_sin",
    "pickup_dow_cos",
]

categorical_input_cols = [
    "pickup_zone_cat",
    "dropoff_zone_cat",
    "pickup_borough_cat",
    "dropoff_borough_cat",
    "vendor_id_cat",
    "rate_code_id_cat",
    "store_and_fwd_flag_cat",
    "weather_condition_code_cat",
]
categorical_index_cols = [f"{column}_index" for column in categorical_input_cols]
categorical_vector_cols = [f"{column}_vector" for column in categorical_input_cols]

print("Continuous features:", continuous_numeric_cols)
print("Binary features:", binary_cols)
print("Cyclical features:", cyclical_cols)
print("Categorical features:", categorical_input_cols)

Continuous features: ['trip_distance', 'passenger_count_imputed', 'pickup_month_of_year', 'weather_temperature_c', 'weather_relative_humidity_pct_imputed', 'weather_precipitation_mm_imputed', 'weather_wind_speed_kmh', 'weather_pressure_hpa', 'air_quality_pm25_mean_imputed']
Binary features: ['is_weekend', 'air_quality_missing']
Cyclical features: ['pickup_hour_sin', 'pickup_hour_cos', 'pickup_dow_sin', 'pickup_dow_cos']
Categorical features: ['pickup_zone_cat', 'dropoff_zone_cat', 'pickup_borough_cat', 'dropoff_borough_cat', 'vendor_id_cat', 'rate_code_id_cat', 'store_and_fwd_flag_cat', 'weather_condition_code_cat']


### 4. Build the reusable Spark feature pipeline

`handleInvalid="keep"` ensures unseen Validation/Test categories receive a valid fallback bucket rather than failing transformation.

In [5]:
prepare_categories = SQLTransformer(
    statement="""
        SELECT
            *,
            COALESCE(pickup_zone, 'Unknown') AS pickup_zone_cat,
            COALESCE(dropoff_zone, 'Unknown') AS dropoff_zone_cat,
            COALESCE(pickup_borough, 'Unknown') AS pickup_borough_cat,
            COALESCE(dropoff_borough, 'Unknown') AS dropoff_borough_cat,
            COALESCE(CAST(vendor_id AS STRING), 'Unknown') AS vendor_id_cat,
            COALESCE(CAST(rate_code_id AS STRING), 'Unknown') AS rate_code_id_cat,
            COALESCE(store_and_fwd_flag, 'Unknown') AS store_and_fwd_flag_cat,
            COALESCE(
                CAST(weather_condition_code AS STRING),
                'Unknown'
            ) AS weather_condition_code_cat
        FROM __THIS__
    """
)

numeric_imputer = Imputer(
    strategy="median",
    inputCols=imputer_input_cols,
    outputCols=imputer_output_cols,
)

category_indexer = StringIndexer(
    inputCols=categorical_input_cols,
    outputCols=categorical_index_cols,
    handleInvalid="keep",
    stringOrderType="alphabetAsc",
)

category_encoder = OneHotEncoder(
    inputCols=categorical_index_cols,
    outputCols=categorical_vector_cols,
    handleInvalid="keep",
    dropLast=True,
)

continuous_assembler = VectorAssembler(
    inputCols=continuous_numeric_cols,
    outputCol="continuous_features_raw",
    handleInvalid="keep",
)

continuous_scaler = RobustScaler(
    inputCol="continuous_features_raw",
    outputCol="continuous_features_scaled",
    lower=0.25,
    upper=0.75,
    withCentering=False,
    withScaling=True,
)

final_assembler = VectorAssembler(
    inputCols=[
        "continuous_features_scaled",
        *binary_cols,
        *cyclical_cols,
        *categorical_vector_cols,
    ],
    outputCol="features",
    handleInvalid="keep",
)

feature_pipeline = Pipeline(
    stages=[
        prepare_categories,
        numeric_imputer,
        category_indexer,
        category_encoder,
        continuous_assembler,
        continuous_scaler,
        final_assembler,
    ]
)

print("Pipeline stages:")
for position, stage in enumerate(feature_pipeline.getStages(), start=1):
    print(f"{position}. {stage.__class__.__name__}")

Pipeline stages:
1. SQLTransformer
2. Imputer
3. StringIndexer
4. OneHotEncoder
5. VectorAssembler
6. RobustScaler
7. VectorAssembler


### 5. Fit on Train only and save the fitted pipeline

The learned medians, category mappings, one-hot dimensions, and robust scaling statistics all come exclusively from Train.

In [6]:
feature_pipeline_model = feature_pipeline.fit(train_raw)

feature_pipeline_model.write().overwrite().save(str(PIPELINE_MODEL_PATH))
print("Fitted feature pipeline saved to:", PIPELINE_MODEL_PATH)


[Stage 38:================>                                      (10 + 10) / 33]




[Stage 38:=================================================>      (29 + 4) / 33]




[Stage 43:===============>                                        (9 + 10) / 33]




[Stage 43:=================================================>      (29 + 4) / 33]




[Stage 48:=======================>                               (14 + 10) / 33]




[Stage 48:===============================================>        (28 + 5) / 33]



Fitted feature pipeline saved to: /Users/chloexia/Documents/KTH/Period 1/Data-Intensive Computing/Labs/data-intensive-computing/data/models/fare_feature_pipeline


### 6. Transform all splits with the same fitted pipeline

In [7]:
output_columns = [
    "trip_id",
    "pickup_datetime",
    "split",
    "label",
    "features",
]

train_features = feature_pipeline_model.transform(train_raw).select(output_columns)
validation_features = feature_pipeline_model.transform(validation_raw).select(output_columns)
test_features = feature_pipeline_model.transform(test_raw).select(output_columns)

feature_dataset = (
    train_features
    .unionByName(validation_features)
    .unionByName(test_features)
)

feature_dataset.printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- split: string (nullable = true)
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)



### 7. Write the final feature Delta table

The table is partitioned by split so Task 3 can load Train, Validation, or Test independently.

In [8]:
(
    feature_dataset.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("split")
    .save(str(FEATURE_OUTPUT_PATH))
)

print("Fare feature Delta table written to:", FEATURE_OUTPUT_PATH)

26/09/29 19:40:53 WARN DAGScheduler: Broadcasting large task binary with size 1168.9 KiB



[Stage 76:=============>                                         (11 + 10) / 45]




[Stage 76:=================>                                     (14 + 10) / 45]




[Stage 76:====================>                                  (17 + 10) / 45]




[Stage 76:=======================>                               (19 + 10) / 45]




[Stage 76:=============================>                         (24 + 11) / 45]




[Stage 76:====================================>                  (30 + 10) / 45]




[Stage 76:=========================================>             (34 + 10) / 45]



Fare feature Delta table written to: /Users/chloexia/Documents/KTH/Period 1/Data-Intensive Computing/Labs/data-intensive-computing/data/processed/ml/fare_feature_dataset


## Checks

Reload the saved table and confirm that row counts and labels are preserved, every split uses one stable feature dimension, and the final vector contains no null or NaN values.

In [9]:
persisted_features = spark.read.format("delta").load(str(FEATURE_OUTPUT_PATH))

output_split_counts = {
    row["split"]: row["count"]
    for row in persisted_features.groupBy("split").count().collect()
}
assert output_split_counts == input_split_counts, (
    f"Row counts changed: input={input_split_counts}, output={output_split_counts}"
)

feature_arrays = persisted_features.withColumn(
    "_feature_array",
    vector_to_array("features"),
)

feature_dimensions = sorted(
    row["feature_dimension"]
    for row in feature_arrays.select(
        F.size("_feature_array").alias("feature_dimension")
    ).distinct().collect()
)
assert len(feature_dimensions) == 1, f"Inconsistent feature sizes: {feature_dimensions}"

invalid_feature_rows = feature_arrays.filter(
    F.exists(
        "_feature_array",
        lambda value: value.isNull() | F.isnan(value),
    )
).count()
assert invalid_feature_rows == 0, f"Found {invalid_feature_rows} rows with invalid features"

quality = persisted_features.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("trip_id").alias("distinct_trip_ids"),
    F.sum(F.col("label").isNull().cast("long")).alias("null_labels"),
    F.sum((F.col("label") < 0).cast("long")).alias("negative_labels"),
).first()

assert quality["row_count"] == quality["distinct_trip_ids"]
assert quality["null_labels"] == 0
assert quality["negative_labels"] == 0

print("Input/output split counts:", output_split_counts)
print("Feature dimension:", feature_dimensions[0])
print("Invalid feature rows:", invalid_feature_rows)
print("Quality:", dict(quality.asDict()))

persisted_features.select(
    "trip_id",
    "pickup_datetime",
    "split",
    "label",
    "features",
).show(5, truncate=100)


[Stage 94:================>                                      (10 + 10) / 33]




[Stage 94:===============================>                       (19 + 10) / 33]




[Stage 94:==================================================>     (30 + 3) / 33]




[Stage 99:==========================>                            (16 + 10) / 33]




[Stage 99:===============================>                       (19 + 10) / 33]




[Stage 99:==========================================>             (25 + 8) / 33]




[Stage 104:=============>                                         (8 + 10) / 33]




[Stage 104:=============================================>         (27 + 6) / 33]




[Stage 106:=================================>                    (20 + 10) / 32]



Input/output split counts: {'train': 9381837, 'test': 306990, 'validation': 167440}
Feature dimension: 561
Invalid feature rows: 0
Quality: {'row_count': 9856267, 'distinct_trip_ids': 9856267, 'null_labels': 0, 'negative_labels': 0}
+----------------------------------------------------------------+-------------------+-----+-----+----------------------------------------------------------------------------------------------------+
|                                                         trip_id|    pickup_datetime|split|label|                                                                                            features|
+----------------------------------------------------------------+-------------------+-----+-----+----------------------------------------------------------------------------------------------------+
|0000625b0b18038c09ef7ac5ca7aadd270f380ac982a4ea8c45965ed7a87fdf6|2024-01-04 15:09:33|train| 11.4|(561,[0,2,3,4,6,7,10,11,12,13,14,161,351,518,524,531,533,542,555],[0.